In [ ]:
# 1) УСТАНОВКА ПАКЕТОВ
#%pip install requests
#%pip install telethon
#%pip install --upgrade ipykernel
#%pip install requests lxml pandas
#%pip install selenium
#%pip install pymongo
#%pip install requests lxml mysql-connector-python
#%pip install webdriver-manager

In [ ]:
# Прежде чем запускать нужно закрыть все окна Chrome (важно для user-data-dir).

# 2) ИМПОРТЫ
import os, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

In [ ]:
# 3) НАСТРОЙКИ
REUSE_SESSION = True
USER_DATA_DIR = os.path.expanduser(r"~\.chrome-selenium-gmail")  # отдельный профиль (Windows-стиль)
MAX_INBOX = 50
MAX_SENT  = 50
IMPLICIT_TIMEOUT = 0
EXPLICIT_TIMEOUT = 30

# Если интернета нет или Selenium Manager не справится,нужно указать путь к локальному драйверу:
LOCAL_DRIVER = None  # например: r"C:\WebDrivers\chromedriver.exe"

# 4) ЗАПУСК ДРАЙВЕРА CHROME (БЕЗ webdriver-manager)
from selenium.common.exceptions import SessionNotCreatedException, WebDriverException

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
# chrome_options.add_argument("--headless=new")  # вход с 2FA в headless обычно невозможен
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)

if REUSE_SESSION:
    os.makedirs(USER_DATA_DIR, exist_ok=True)
    chrome_options.add_argument(f"--user-data-dir={USER_DATA_DIR}")
    chrome_options.add_argument("--profile-directory=Default")

def build_driver():
    # 1) Если указан локальный драйвер — используем его
    if LOCAL_DRIVER:
        service = Service(executable_path=LOCAL_DRIVER)
        return webdriver.Chrome(service=service, options=chrome_options)

    # 2) Иначе пусть Selenium Manager сам подберёт драйвер (нужен интернет при первом запуске)
    try:
        return webdriver.Chrome(options=chrome_options)
    except SessionNotCreatedException as e:
        print("⚠️ SessionNotCreatedException: пробуем запустить с чистым профилем (без user-data-dir).")
        # 3) Резерв: чистый профиль (часто помогает, если предыдущий профиль «залочен»/повреждён)
        clean_opts = Options()
        clean_opts.add_argument("--start-maximized")
        clean_opts.add_argument("--disable-blink-features=AutomationControlled")
        clean_opts.add_experimental_option("excludeSwitches", ["enable-automation"])
        clean_opts.add_experimental_option("useAutomationExtension", False)
        try:
            return webdriver.Chrome(options=clean_opts)
        except Exception:
            raise
    except WebDriverException as e:
        # Если Selenium Manager не может скачать драйвер (офлайн) — предложим использовать LOCAL_DRIVER
        raise RuntimeError(
            "Не удалось запустить ChromeDriver через Selenium Manager. "
            "Если офлайн, нужно указать путь к локальному драйверу в LOCAL_DRIVER."
        ) from e

driver = build_driver()
driver.implicitly_wait(IMPLICIT_TIMEOUT)
wait = WebDriverWait(driver, EXPLICIT_TIMEOUT)


In [ ]:
# 5) ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def ensure_gmail_loaded(target="#inbox"):
    """Открываем Gmail, ждём нужную метку или логин (вход руками), затем наличие строк писем."""
    url = f"https://mail.google.com/mail/u/0/{target}"
    driver.get(url)

    # Fallback для #sent: иногда прямой URL не прорисовывает список — кликаем пункт слева
    if target == "#sent":
        try:
            driver.find_element(By.CSS_SELECTOR, 'a[href$="#sent"]').click()
        except Exception:
            pass

    # ждём либо строки, либо логин
    try:
        wait.until(EC.any_of(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="main"] tr.zA')),
            EC.presence_of_element_located((By.CSS_SELECTOR, "tr.zA")),
            EC.url_contains("accounts.google.com"),
            EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='main']"))
        ))
    except TimeoutException:
        pass

    # если логин
    if "accounts.google.com" in driver.current_url:
        print("⚠️ Открылся вход в Google. Завершите вход (пароль, 2FA, подтверждение).")
        print("После успешной авторизации Gmail откроется автоматически. Если зависло — нужно перезапустить эту ячейку.")
        try:
            wait.until(EC.url_contains("mail.google.com"))
            # после входа — убедимся, что открыта нужная метка
            if target == "#sent":
                try:
                    driver.find_element(By.CSS_SELECTOR, 'a[href$="#sent"]').click()
                except Exception:
                    pass
        except TimeoutException:
            raise RuntimeError("Не удалось авторизоваться. Повтори запуск ячейки после ручного входа.")

    # убеждаемся, что мы действительно на нужной метке
    try:
        wait.until(EC.url_contains(target))
    except TimeoutException:
        # ещё раз ткнёмся в пункт меню для #sent и подождём URL
        if target == "#sent":
            try:
                driver.find_element(By.CSS_SELECTOR, 'a[href$="#sent"]').click()
                wait.until(EC.url_contains(target))
            except Exception:
                pass

    # финально ждём наличие строк
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="main"] tr.zA')))
    except TimeoutException:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tr.zA")))

def _scroll_to_load(min_rows=50, max_scrolls=20, pause=0.7):
    """это прокрутка, чтобы Gmail подгрузил больше тредов."""
    for _ in range(max_scrolls):
        rows = driver.find_elements(By.CSS_SELECTOR, 'div[role="main"] tr.zA')
        if not rows:
            rows = driver.find_elements(By.CSS_SELECTOR, "tr.zA")
        if len(rows) >= min_rows:
            return
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

def _txt(el, sel, attr=None):
    """Безопасно достаем текст/атрибут у потомка el по селектору sel."""
    try:
        child = el.find_element(By.CSS_SELECTOR, sel)
        return child.get_attribute(attr) if attr else child.text
    except NoSuchElementException:
        return None
    except StaleElementReferenceException:
        return None

def _parse_row(row):
    
    thread_id = _txt(row, "span[data-legacy-thread-id]", attr="data-legacy-thread-id") or ""
    subject = _txt(row, "span.bog") or "" #Пытаемся взять тему письма из вложенного элемента span.bog
    snippet = (_txt(row, "span.y2") or "").lstrip(" - ").strip() #Берём “кусочек содержания” письма из span.y2 и убираем дефис и пробелы по бокам 

    date_title = _txt(row, "td.xW span", attr="title") #Берём атрибут title у span в колонке даты
    date_visible = _txt(row, "td.xW span") #Берём видимый текст той же даты
    date_str = date_title or date_visible or ""

    contact_el = None
    for sel in ("span.yP[email]", "div.afn span[email]", "span[email]"):
        els = row.find_elements(By.CSS_SELECTOR, sel)
        if els:
            contact_el = els[0]
            break

    if contact_el:
        email = (contact_el.get_attribute("email") or "").strip()
        name = (contact_el.get_attribute("name") or contact_el.text or "").strip()
    else:
        email, name = "", ""

    return {
        "contact_email": email,
        "contact_name": name,
        "subject": subject,
        "snippet": snippet,
        "date": date_str,
        "thread_id": thread_id
    }

def fetch_label(label_hash="#inbox", limit=50):
    """Собираем limit тредов из заданной метки (#inbox / #sent). Возвращает DataFrame."""
    ensure_gmail_loaded(label_hash)
    # ждём появление хотя бы одной строки в основной области
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="main"] tr.zA')))
    except TimeoutException:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tr.zA")))

    _scroll_to_load(min_rows=limit)
    rows = driver.find_elements(By.CSS_SELECTOR, 'div[role="main"] tr.zA')
    if not rows:
        rows = driver.find_elements(By.CSS_SELECTOR, "tr.zA")
    rows = rows[:limit]

    data = [_parse_row(r) for r in rows]
    return pd.DataFrame(data)

# 6) СБОР ДАННЫХ
inbox_df = fetch_label("#inbox", limit=MAX_INBOX)
sent_df  = fetch_label("#sent",  limit=MAX_SENT)

In [11]:
# 7) РЕЗУЛЬТАТЫ
print(f"Собрано из Входящих: {len(inbox_df)}")
display(inbox_df.head(20))
print(f"Собрано из Отправленных: {len(sent_df)}")
display(sent_df.head(20))

Собрано из Входящих: 50


,contact_email,contact_name,subject,snippet,date,thread_id
0,no-reply@accounts.google.com,Google,Ваш аккаунт Google восстановлен,Ваш аккаунт восстановлен iliya.bendik1@gmail.c...,"ср, 24 дек. 2025 г., 16:12",19b500fa20157827
1,noreply@stepik.org,Kanta Kanta,Strangers Things 5. Episode 4,"Hi there, Илья Юрьевич! На курсе вышел 4-ый, н...","пн, 22 дек. 2025 г., 15:40",19b45a57fba2c7d8
2,no-reply@psu.ru,no-reply,пропуски занятий,Добрый день! В связи с поступлением большого к...,"пн, 22 дек. 2025 г., 14:26",19b45614044ffde9
3,noreply@stepik.org,Stepik Team,"Отличная была неделя! Ваш прогресс в курсе """"П...",,"пн, 22 дек. 2025 г., 10:56",19b44a10a91cb772
4,noreply@taxcom.ru,ОФД Такском,Кассовый чек от Общество с ограниченной ответс...,,"вс, 21 дек. 2025 г., 15:03",19b405ce81ec2d99
5,notification@service.citilink.ru,Ситилинк,Ваш заказ принят,Благодарим за покупку! Не открываются картинки...,"вс, 21 дек. 2025 г., 14:59",19b4059f8de279b2
6,iliya.bendik1@gmail.com,я,Уточнение по НИР,Здравствуйте! Не совсем так - научный руководи...,"пт, 19 дек. 2025 г., 23:08",19b362eddd2c421d
7,DoNotReply@notify.orcid.org,ORCID - Do not reply,[ORCID] Iliya Bendik у вас новые уведомления,Здравствуйте! Iliya Bendik (https://orcid.org/...,"пт, 19 дек. 2025 г., 17:43",19b36a2a6b355fc7
8,noreply@stepik.org,Иосиф Дзеранов,Запись эфира про мотивацию,"Илья Юрьевич, спасибо, что был с нами на эфире...","пт, 19 дек. 2025 г., 16:38",19b3667afd622c25
9,noreply@billing.dom.ru,Дом.ру,Чек об оплате услуг ДОМ.РУ,͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌ ﻿͏‌...,"пт, 19 дек. 2025 г., 11:45",19b355b980b363c1


Собрано из Отправленных: 50


,contact_email,contact_name,subject,snippet,date,thread_id
0,dmakarov@psu.ru,dmakarov,Уточнение по НИР,"Добрый день, я студент группы ПМИ-2-2024 НМ ФМ...","пт, 19 дек. 2025 г., 15:42",19b362eddd2c421d
1,marina@barulina.ru,marina,Финальный вариант - Сводка n8n_Илья Бендик,How to Do Great Work **Concise Summary** - **F...,"вт, 4 нояб. 2025 г., 23:39",19a502b1dabbd7f7
2,ikochkareva@yandex.ru,irina,контрольные работы,"Хорошо, спасибо большое пн, 27 окт. 2025 г., 1...","пн, 27 окт. 2025 г., 19:19",195d616e5f29ae89
3,psu501@yandex.ru,Людмила,Уточнение по первой контрольной точке,"Прикладываю выполненное задание по 3КТ пт, 24 ...","пт, 24 окт. 2025 г., 14:08",1951dda21b9f04e0
4,mbuzmakova@psu.ru,mbuzmakova,Отчет по задачам Бендик Илья,"ХОРОШО поставила, приносите экзаменационный ли...","вт, 21 окт. 2025 г., 21:32",19a079b86b7e1d10
5,ilinvladimir1@gmail.com,Владимир,Исправление долга,"Хорошо пн, 20 окт. 2025 г., 20:13 Владимир Иль...","пн, 20 окт. 2025 г., 20:13",1993da1d0c4bf09e
6,ikochkareva@yandex.ru,irina,English files,"Добрый вечер, могу ли я вас попросить отправит...","пн, 15 сент. 2025 г., 18:25",19464c224926d1b7
7,dn_niko@mail.ru,Николай,Форма_Согласие на обработку персональных данны...,"Добрый день, отправляю вам заполненное согласи...","пт, 1 авг. 2025 г., 12:30",19860f9d4d5a4bd2
8,unsubscription@ozon.ru,unsubscript.,unsubscribe:J45q0_5B7AQ8Ws5JWdpobCncWP-lmbH3KG...,,"пн, 14 апр. 2025 г., 18:12",196346de77b0b4ce
9,newsletter@ostrovok.ru,newsletter,unsubscribe,This message was automatically generated by Gm...,"пн, 7 апр. 2025 г., 16:50",1961016e84538cc5
